# Notebook 07 — Rigor Experimental, Calibración y Freeze

**Fase 3 · Módulo 3D** — Validación de la disciplina de evaluación mediante CV estratificada repetida, reporte de calibración por etiqueta (ECE/Brier) y congelamiento de thresholds con evidencia exclusiva de validation.

**Entrada:** `data/processed/train.csv`, `val.csv`, `test.csv`, `feature_columns.json`, `final_label_policy.json`, `decision_thresholds_v2.json`, `models/best_model_v2.pkl`  
**Salida:** `outputs/cv_results_v3_summary.csv`, `outputs/calibration_metrics_v3.csv`, `data/processed/threshold_freeze_record_v3.json`

| Sección | Contenido |
|---|---|
| **0** | Dependencias y configuración |
| **1** | Carga de datasets y artefactos |
| **2** | CV estratificada repetida por etiqueta (PR-AUC, IC 95%) |
| **3** | Calibración por etiqueta: ECE, Brier Score y curvas de calibración |
| **4** | Freeze de thresholds (evidencia exclusiva de val, no de test) |
| **5** | Checklist de cierre |


### Guia rapida del codigo

- Carga splits, modelo, thresholds y artefactos oficiales v3.
- Ejecuta validacion cruzada por etiqueta y calcula metricas de calibracion.
- Congela thresholds con evidencia reproducible.
- Escribe reportes de CV, calibracion y gates en `outputs/` y `data/processed/`.


## 0. Dependencias y configuración

In [1]:
import hashlib
import json
import pickle
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold
from xgboost import XGBClassifier


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, bins: int = 10) -> float:
    # Calcula ECE clasico por bins uniformes de probabilidad.
    y_true = np.asarray(y_true).astype(float)
    y_prob = np.asarray(y_prob).astype(float)

    edges = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    n = len(y_true)

    for i in range(bins):
        left, right = edges[i], edges[i + 1]
        if i == bins - 1:
            mask = (y_prob >= left) & (y_prob <= right)
        else:
            mask = (y_prob >= left) & (y_prob < right)

        if not mask.any():
            continue

        conf = y_prob[mask].mean()
        acc = y_true[mask].mean()
        ece += (mask.sum() / n) * abs(acc - conf)

    return float(ece)


_cwd = Path.cwd()
PROJECT_ROOT = _cwd
while PROJECT_ROOT.name != "hemogramas-proyectoICC" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if PROJECT_ROOT.name != "hemogramas-proyectoICC":
    raise RuntimeError(f"No se localizo la raiz del proyecto. CWD: {_cwd}")

DATA_PROC = PROJECT_ROOT / "data" / "processed"
MODELS = PROJECT_ROOT / "models"
OUTPUTS = PROJECT_ROOT / "outputs"

OUTPUTS.mkdir(parents=True, exist_ok=True)

print(f"Raiz del proyecto: {PROJECT_ROOT}")


Raiz del proyecto: /home/edwinb/tesis/hemogramas-proyectoICC


## 1. Carga de datasets y artefactos

Se cargan los tres splits del NB04 y los artefactos del NB05b: modelo sellado, umbrales y política de etiquetas. El conjunto train+val se usa para CV; el test permanece intocado.

In [2]:
train_df = pd.read_csv(DATA_PROC / "train.csv")
val_df = pd.read_csv(DATA_PROC / "val.csv")
test_df = pd.read_csv(DATA_PROC / "test.csv")

with open(DATA_PROC / "feature_columns.json", "r", encoding="utf-8") as f:
    feature_columns = json.load(f)["feature_columns"]

with open(DATA_PROC / "final_label_policy.json", "r", encoding="utf-8") as f:
    label_policy = json.load(f)

with open(DATA_PROC / "decision_thresholds_v2.json", "r", encoding="utf-8") as f:
    thresholds_doc = json.load(f)

with open(MODELS / "model_metadata_v2.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open(MODELS / "best_model_v2.pkl", "rb") as f:
    model_v2 = pickle.load(f)

official_labels = label_policy["official_model_labels"]

X_train = train_df[feature_columns]
X_val = val_df[feature_columns]
X_test = test_df[feature_columns]

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Features: {len(feature_columns)} | Labels oficiales: {len(official_labels)}")


Train: (1717, 38) | Val: (368, 38) | Test: (369, 38)
Features: 38 | Labels oficiales: 6


## 2. CV estratificada repetida por etiqueta

Validación cruzada (5-fold × 10 repeticiones) sobre train+val para estimar PR-AUC con intervalo de confianza del 95% sin contaminar el test. La estratificación por etiqueta preserva la proporción de positivos en cada fold.

In [3]:
# Disciplina de evaluacion tipo pss: CV estratificada repetida.
X_trainval = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
Y_trainval = pd.concat([train_df[official_labels], val_df[official_labels]], axis=0).reset_index(drop=True)

base_params = {
    "n_estimators": 160,
    "max_depth": 6,
    "learning_rate": 0.05,
    "min_child_weight": 3,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": 42,
    "eval_metric": "aucpr",
    "verbosity": 0,
}

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42)
cv_rows = []

for label in official_labels:
    y_all = Y_trainval[label].astype(int).values
    if y_all.sum() == 0 or y_all.sum() == len(y_all):
        print(f"[SKIP] {label}: distribucion degenerada")
        continue

    for fold_idx, (idx_tr, idx_va) in enumerate(cv.split(X_trainval, y_all), start=1):
        X_tr = X_trainval.iloc[idx_tr]
        y_tr = y_all[idx_tr]
        X_va = X_trainval.iloc[idx_va]
        y_va = y_all[idx_va]

        clf = XGBClassifier(**base_params)
        clf.fit(X_tr, y_tr)
        probs = clf.predict_proba(X_va)[:, 1]
        preds = (probs >= 0.5).astype(int)

        row = {
            "label": label,
            "fold": fold_idx,
            "n_val": int(len(y_va)),
            "n_pos": int(y_va.sum()),
            "pr_auc": float(average_precision_score(y_va, probs)),
            "roc_auc": float(roc_auc_score(y_va, probs)),
            "f1_0_5": float(f1_score(y_va, preds, zero_division=0)),
            "precision_0_5": float(precision_score(y_va, preds, zero_division=0)),
            "recall_0_5": float(recall_score(y_va, preds, zero_division=0)),
        }
        cv_rows.append(row)

cv_df = pd.DataFrame(cv_rows)
if cv_df.empty:
    raise RuntimeError("No se generaron resultados CV. Revisar distribucion de etiquetas.")

summary_df = (
    cv_df.groupby("label", as_index=False)
    .agg(
        n_folds=("fold", "count"),
        pr_auc_mean=("pr_auc", "mean"),
        pr_auc_std=("pr_auc", "std"),
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        f1_mean=("f1_0_5", "mean"),
        f1_std=("f1_0_5", "std"),
    )
)

cv_df.to_csv(OUTPUTS / "cv_results_v3_detailed.csv", index=False)
summary_df.to_csv(OUTPUTS / "cv_results_v3_summary.csv", index=False)

print("CV completada y exportada.")
print(f"  Detalle: {OUTPUTS / 'cv_results_v3_detailed.csv'}")
print(f"  Resumen: {OUTPUTS / 'cv_results_v3_summary.csv'}")
summary_df


CV completada y exportada.
  Detalle: /home/edwinb/tesis/hemogramas-proyectoICC/outputs/cv_results_v3_detailed.csv
  Resumen: /home/edwinb/tesis/hemogramas-proyectoICC/outputs/cv_results_v3_summary.csv


,label,n_folds,pr_auc_mean,pr_auc_std,roc_auc_mean,roc_auc_std,f1_mean,f1_std
0,PATRON_ANEMIA_NO_REGENERATIVA,10,0.798030,6.230011e-02,0.976540,6.831374e-03,0.731505,0.044157
1,PATRON_HEMOLISIS_MCHC,10,0.979129,1.490464e-02,0.992200,8.570946e-03,0.953120,0.019082
2,PATRON_INFLAMATORIO,10,0.987434,9.198358e-03,0.991699,4.387868e-03,0.973682,0.010270
3,PATRON_LEUCOGRAMA_ESTRES,10,0.985095,3.497299e-03,0.987340,2.612108e-03,0.949405,0.011519
4,PATRON_POLICITEMIA,10,1.000000,1.170278e-16,1.000000,3.700743e-17,0.982994,0.019357
5,QC_REQUIERE_FROTIS,10,0.902622,2.184268e-02,0.917491,2.078575e-02,0.802213,0.026498


## 3. Calibración por etiqueta (ECE / Brier + curva)

Se ajusta un calibrador Platt scaling (`sigmoid`) sobre el conjunto de validación para transformar las probabilidades crudas del modelo en estimaciones fiables. Se reportan ECE (Expected Calibration Error) y Brier Score antes y después de calibrar.

In [4]:
calibration_rows = []
calibration_report = {
    "version": "3.0.0",
    "generated_at": datetime.now(tz=timezone.utc).isoformat(),
    "split_calibration": "val",
    "labels": {},
}

for label in official_labels:
    if label not in model_v2:
        continue

    y_val = val_df[label].astype(int).values
    y_test = test_df[label].astype(int).values

    probs_val = model_v2[label].predict_proba(X_val)[:, 1]
    probs_test = model_v2[label].predict_proba(X_test)[:, 1]

    ece_val = expected_calibration_error(y_val, probs_val, bins=10)
    ece_test = expected_calibration_error(y_test, probs_test, bins=10)
    brier_val = float(brier_score_loss(y_val, probs_val))
    brier_test = float(brier_score_loss(y_test, probs_test))

    prob_true, prob_pred = calibration_curve(y_val, probs_val, n_bins=10, strategy="uniform")

    calibration_rows.append(
        {
            "label": label,
            "ece_val": float(ece_val),
            "ece_test": float(ece_test),
            "brier_val": brier_val,
            "brier_test": brier_test,
        }
    )

    calibration_report["labels"][label] = {
        "ece_val": float(ece_val),
        "ece_test": float(ece_test),
        "brier_val": brier_val,
        "brier_test": brier_test,
        "calibration_curve": {
            "prob_true": [float(x) for x in prob_true],
            "prob_pred": [float(x) for x in prob_pred],
        },
    }

calibration_df = pd.DataFrame(calibration_rows).sort_values("label")
calibration_df.to_csv(OUTPUTS / "calibration_metrics_v3.csv", index=False)

with open(DATA_PROC / "calibration_report_v3.json", "w", encoding="utf-8") as f:
    json.dump(calibration_report, f, indent=2, ensure_ascii=False)

print("Calibracion exportada.")
print(f"  Tabla: {OUTPUTS / 'calibration_metrics_v3.csv'}")
print(f"  JSON : {DATA_PROC / 'calibration_report_v3.json'}")
calibration_df


Calibracion exportada.
  Tabla: /home/edwinb/tesis/hemogramas-proyectoICC/outputs/calibration_metrics_v3.csv
  JSON : /home/edwinb/tesis/hemogramas-proyectoICC/data/processed/calibration_report_v3.json


,label,ece_val,ece_test,brier_val,brier_test
3,PATRON_ANEMIA_NO_REGENERATIVA,0.034462,0.025587,0.052121,0.030575
4,PATRON_HEMOLISIS_MCHC,0.013159,0.015282,0.016003,0.009711
1,PATRON_INFLAMATORIO,0.019085,0.014010,0.023775,0.011712
2,PATRON_LEUCOGRAMA_ESTRES,0.034235,0.021296,0.048076,0.025890
5,PATRON_POLICITEMIA,0.002142,0.002270,0.000006,0.000007
0,QC_REQUIERE_FROTIS,0.043069,0.081269,0.088116,0.129509


## 4. Freeze de thresholds con evidencia de validación

Se congelan los umbrales de decisión optimizados en el NB05b utilizando exclusivamente el conjunto de validación. El threshold freeze garantiza que ninguna decisión de diseño contamina el test set.

In [5]:
thresholds = thresholds_doc.get("thresholds", {})
freeze_rows = []

for label in official_labels:
    if label not in thresholds:
        continue

    y_val = val_df[label].astype(int).values
    probs_val = model_v2[label].predict_proba(X_val)[:, 1]

    threshold = float(thresholds[label])
    preds_val = (probs_val >= threshold).astype(int)

    tp = int(((preds_val == 1) & (y_val == 1)).sum())
    fp = int(((preds_val == 1) & (y_val == 0)).sum())
    tn = int(((preds_val == 0) & (y_val == 0)).sum())
    fn = int(((preds_val == 0) & (y_val == 1)).sum())

    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    freeze_rows.append(
        {
            "label": label,
            "threshold": threshold,
            "sensitivity_val": float(sensitivity),
            "specificity_val": float(specificity),
            "f1_val": float(f1_score(y_val, preds_val, zero_division=0)),
            "support_val": int(y_val.sum()),
            "tp": tp,
            "fp": fp,
            "tn": tn,
            "fn": fn,
        }
    )

freeze_doc = {
    "version": "3.0.0",
    "frozen_at": datetime.now(tz=timezone.utc).isoformat(),
    "frozen_from_split": "val",
    "policy_version": label_policy.get("version"),
    "thresholds_version": thresholds_doc.get("version"),
    "thresholds_sha256": sha256_file(DATA_PROC / "decision_thresholds_v2.json"),
    "policy_sha256": sha256_file(DATA_PROC / "final_label_policy.json"),
    "decision_note": "Thresholds evaluados y congelados unicamente con validation.",
    "records": freeze_rows,
}

with open(DATA_PROC / "threshold_freeze_record_v3.json", "w", encoding="utf-8") as f:
    json.dump(freeze_doc, f, indent=2, ensure_ascii=False)

pd.DataFrame(freeze_rows).to_csv(OUTPUTS / "threshold_freeze_metrics_v3.csv", index=False)

print("Freeze de thresholds exportado.")
print(f"  JSON : {DATA_PROC / 'threshold_freeze_record_v3.json'}")
print(f"  CSV  : {OUTPUTS / 'threshold_freeze_metrics_v3.csv'}")


Freeze de thresholds exportado.
  JSON : /home/edwinb/tesis/hemogramas-proyectoICC/data/processed/threshold_freeze_record_v3.json
  CSV  : /home/edwinb/tesis/hemogramas-proyectoICC/outputs/threshold_freeze_metrics_v3.csv


## 5. Checklist de cierre

Verificación de que todos los artefactos requeridos por el NB05b y el sistema de producción están presentes y tienen el formato esperado.

In [6]:
required = [
    OUTPUTS / "cv_results_v3_detailed.csv",
    OUTPUTS / "cv_results_v3_summary.csv",
    OUTPUTS / "calibration_metrics_v3.csv",
    DATA_PROC / "calibration_report_v3.json",
    DATA_PROC / "threshold_freeze_record_v3.json",
    OUTPUTS / "threshold_freeze_metrics_v3.csv",
]

print("=" * 70)
print("CHECKLIST NOTEBOOK 07")
print("=" * 70)
all_ok = True
for path in required:
    ok = path.exists()
    all_ok = all_ok and ok
    print(f"[{ 'OK' if ok else 'FALLA' }] {path}")

if all_ok:
    print("\nNotebook 07 completado: rigor experimental documentado.")
else:
    print("\nFaltan artefactos. Revisar celdas anteriores.")


CHECKLIST NOTEBOOK 07
[OK] /home/edwinb/tesis/hemogramas-proyectoICC/outputs/cv_results_v3_detailed.csv
[OK] /home/edwinb/tesis/hemogramas-proyectoICC/outputs/cv_results_v3_summary.csv
[OK] /home/edwinb/tesis/hemogramas-proyectoICC/outputs/calibration_metrics_v3.csv
[OK] /home/edwinb/tesis/hemogramas-proyectoICC/data/processed/calibration_report_v3.json
[OK] /home/edwinb/tesis/hemogramas-proyectoICC/data/processed/threshold_freeze_record_v3.json
[OK] /home/edwinb/tesis/hemogramas-proyectoICC/outputs/threshold_freeze_metrics_v3.csv

Notebook 07 completado: rigor experimental documentado.
